In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [ ]:

# root_dir = r'Y:\ZHL\isds\PS\task0808'
root_dir = r'E:\data\202502_signboard\data_annotation\ps_data\task0829'
merge_dir = os.path.join(root_dir, 'merge_dir')
root_folder_id = '1_ZTinfG4j2NDxl_Sz7iZ9dWU3vmYLE2F'
client_secret = r"E:\data\202502_signboard\data_annotation\docs\client_secret.json"
token_path = r'E:\repository\dataset_tools\isds_tool\PS_data\token.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

slam_root_folder_id = '1T1fiU0cIXR5lVgubqL-MXr76DB9fhbFp'
gap_num = 3

In [ ]:
# os.remove(token_path)

In [ ]:
import os
import io
from concurrent.futures import ThreadPoolExecutor
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request


def authenticate_with_google(token_path, client_secret_path):
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secret_path, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token_file:
            token_file.write(creds.to_json())

    service = build('drive', 'v3', credentials=creds)
    return service


def download_large_file(service, file_id, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if os.path.exists(file_path):
        print(f"⚠️ 已存在，跳过: {file_path}")
        return
    print(f"⬇️ Downloading {file_path}")
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(file_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"⬇️ Downloading {file_path}: {int(status.progress() * 100)}%")
    print(f"✅ Finished: {file_path}")

def download_folder_recursive(service, folder_id, save_path):
    os.makedirs(save_path, exist_ok=True)
    query = f"'{folder_id}' in parents and trashed = false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = results.get('files', [])

    for item in items:
        file_id = item['id']
        file_name = item['name']
        file_mime = item['mimeType']
        full_path = os.path.join(save_path, file_name)

        if file_mime == 'application/vnd.google-apps.folder':
            download_folder_recursive(service, file_id, full_path)
        else:
            download_large_file(service, file_id, full_path)

def download_subfolder_task(folder_obj, root_save_path, token_path, client_secret_path):
    # 每个线程都单独认证，避免多线程共享service导致问题
    service = authenticate_with_google(token_path, client_secret_path)
    folder_id = folder_obj['id']
    folder_name = folder_obj['name']
    target_path = os.path.join(root_save_path, folder_name)
    print(f"\n📁 Starting folder: {folder_name}")
    download_folder_recursive(service, folder_id, target_path)

def download_all_subfolders_parallel(token_path, client_secret_path, root_folder_id, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    # 主线程先获取子文件夹列表
    service = authenticate_with_google(token_path, client_secret_path)
    query = f"'{root_folder_id}' in parents and trashed = false and mimeType = 'application/vnd.google-apps.folder'"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    folders = results.get('files', [])

    print(f"将并发下载 {len(folders)} 个子文件夹...\n")

    with ThreadPoolExecutor(max_workers=len(folders)) as executor:
        for folder in folders:
            executor.submit(download_subfolder_task, folder, save_dir, token_path, client_secret_path)



In [5]:
download_all_subfolders_parallel(token_path, client_secret, root_folder_id, root_dir)

将并发下载 2 个子文件夹...


📁 Starting folder: ssp2

📁 Starting folder: ssp1
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\rectified_images.zip
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\rectified_images.zip
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\rectified_images.zip: 0%
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\rectified_images.zip: 0%
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\rectified_images.zip: 0%
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\rectified_images.zip: 0%
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\rectified_images.zip: 1%
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\rectified_images.zip: 0%
⬇️ Downloading E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\rectified_images.zip: 1%
⬇️ Downloadin

In [6]:
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

def process_dirs(root_dir):
    sub_dirs = os.listdir(root_dir)
    for idx, sub_name in enumerate(sub_dirs):
        sub_dir = os.path.join(root_dir, sub_name)
        if not os.path.isdir(sub_dir) or sub_name.startswith('r'):
            continue
        cam_name_list = ['camera1', 'camera2', 'camera3', 'camera4', 'camera5', 'camera6']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, cam_name, 'raw')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                print(f'{image_dir_src} selecting...')
                image_dir_select = image_dir_src+'_select'
                shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
                select_img(image_dir_src, image_dir_select, gap=gap_num)
                print(f'{image_dir_select} filtering...')
                image_dir_filter = image_dir_src+'_filter'
                shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
                filter_deduplication(image_dir_select, image_dir_filter)
                print(f'{image_dir_filter} done\n')

In [7]:
process_dirs(root_dir)

E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera1\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera2\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera3\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera4\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera5\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera6\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera1\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera2\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera3\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera4\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera5\raw not exists
E:\data\202502_signboard\data_annotation\ps_data\task0

In [8]:
def img_merge(input_dir, output_dir):
    sub_dirs = os.listdir(input_dir)
    if 'merge_dir' in sub_dirs:
        sub_dirs.remove('merge_dir')
    os.makedirs(output_dir, exist_ok=True)
    for sub_name in sub_dirs:
        sub_dir = os.path.join(input_dir, sub_name)
        if not os.path.isdir(sub_dir) or sub_name.startswith('r'):
            continue
        cam_name_list = ['camera1', 'camera2', 'camera3', 'camera4', 'camera5', 'camera6']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, cam_name, 'raw_filter')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                img_list = os.listdir(image_dir_src)
                for img_name in tqdm(img_list):
                    img_path_src = os.path.join(image_dir_src, img_name)
                    img_path_dst = os.path.join(output_dir, cam_name+'_'+img_name)
                    shutil.copyfile(img_path_src, img_path_dst)



In [9]:
img_merge(root_dir, merge_dir)

E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera1\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera2\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera3\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera4\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera5\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp1\camera6\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera1\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera2\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera3\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera4\raw_filter not exists
E:\data\202502_signboard\data_annotation\ps_data\task0829\ssp2\camera5

In [10]:
print(len(os.listdir(merge_dir)))

0


In [11]:
import zipfile
import os

def zip_folder_to_path(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"zip '{source_folder}' to '{destination_zip}'")

zip_folder_to_path(
    source_folder=merge_dir,
    destination_zip=os.path.join(root_dir, os.path.basename(root_dir)+'.zip')
)

zip 'E:\data\202502_signboard\data_annotation\ps_data\task0829\merge_dir' to 'E:\data\202502_signboard\data_annotation\ps_data\task0829\task0829.zip'
